# Collect a diverse multi-modal dataset

Collect random-policy rollouts from several Gymnasium environments — spanning every
observation and action modality — into a single `DatasetStore`, following the
[mouse-env rollout contract](https://github.com/micahr234/mouse-env/blob/main/docs/guide.md).

| Environment | Observation | Action |
|---|---|---|
| `FrozenLake-v1`, `Taxi-v4` | discrete | discrete |
| `CartPole-v1` | continuous | discrete |
| `Pendulum-v1` | continuous | continuous |
| `ALE/Breakout-v5` | image (84×84) | discrete |

One store holds them all: each modality becomes its own tensor (`obs_discrete` /
`obs_continuous` / `obs_image`, and `action` / `action_continuous`), zero-filled for
steps where that modality is absent. The store's `max_*` dimensions are sized to the
largest values across the environments.

> **Note:** continuous actions are stored in the dataset, but the current MOUSE model is
> discrete-action only — `action_continuous` is carried for completeness / future use.

Requires the examples + atari extras: `pip install -e ".[all]"`.

In [ ]:
import numpy as np
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing

from mouse_core.data import DatasetStore, push_stores_to_hub

gym.register_envs(ale_py)  # register the ALE/* Atari environments

IMG_SIZE = 84  # Atari frames are preprocessed to IMG_SIZE x IMG_SIZE grayscale

# Environments spanning every observation/action modality. They are collected into
# one store; modalities absent from a given step are zero-filled at encode time.
ENV_SPECS = [
    {"id": "FrozenLake-v1",   "kwargs": {"is_slippery": True}, "obs": "discrete",   "action": "discrete"},
    {"id": "Taxi-v4",         "kwargs": {},                    "obs": "discrete",   "action": "discrete"},
    {"id": "CartPole-v1",     "kwargs": {},                    "obs": "continuous", "action": "discrete"},
    {"id": "Pendulum-v1",     "kwargs": {},                    "obs": "continuous", "action": "continuous"},
    {"id": "ALE/Breakout-v5", "kwargs": {"frameskip": 1},      "obs": "image",      "action": "discrete",
     "atari": True, "episodes": 3, "max_steps": 50},
]
EPISODES_PER_ENV = 10
MAX_STEPS = 100

## Collect rollouts

For each environment, encode the observation and action into their typed-dict modality
and append to the same store. Each token at step `t` carries the action and reward that
*produced* `obs_t`, and the integer-coded `done` marks episode boundaries (`0` running,
`1` terminated, `2` truncated).

In [ ]:
def make_env(spec):
    env = gym.make(spec["id"], **spec["kwargs"])
    if spec.get("atari"):
        env = AtariPreprocessing(env, grayscale_obs=True, scale_obs=False, screen_size=IMG_SIZE)
    return env


def encode_obs(obs, kind):
    if kind == "discrete":
        return {"discrete": int(obs)}
    if kind == "continuous":
        return {"continuous": np.asarray(obs, dtype=np.float32).ravel().tolist()}
    return {"image": np.asarray(obs).astype(np.int32).ravel().tolist()}


def encode_action(action, kind):
    if kind == "discrete":
        return {"discrete": int(action)}
    return {"continuous": np.asarray(action, dtype=np.float32).ravel().tolist()}


def collect_episodes(store, env, spec, num_episodes, max_steps):
    """Append random-policy rollout steps from `env` into `store`, by modality."""
    obs_kind, act_kind = spec["obs"], spec["action"]
    for _ in range(num_episodes):
        obs, _ = env.reset()
        # The action stored at step t is the one that produced obs_t; seed step 0.
        action = {"discrete": 0} if act_kind == "discrete" else {"continuous": [0.0]}
        reward = 0.0
        done_flag = 0

        for step_idx in range(max_steps):
            store.append({
                "observation": encode_obs(obs, obs_kind),
                "action": action,
                "reward": reward,
                "done": done_flag,
                "time": step_idx,
            })

            raw_action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(raw_action)
            action = encode_action(raw_action, act_kind)
            reward = float(reward)
            done_flag = 1 if terminated else (2 if truncated else 0)

            if terminated or truncated:
                store.append({
                    "observation": encode_obs(obs, obs_kind),
                    "action": action,
                    "reward": reward,
                    "done": done_flag,
                    "time": step_idx + 1,
                })
                break


store = DatasetStore(
    max_action_dim=18,
    max_action_continuous_dim=1,
    max_obs_continuous_dim=6,
    max_obs_discrete_dim=1,
    max_obs_image_pixels=IMG_SIZE * IMG_SIZE,
)

for spec in ENV_SPECS:
    env = make_env(spec)
    collect_episodes(store, env, spec, spec.get("episodes", EPISODES_PER_ENV), spec.get("max_steps", MAX_STEPS))
    env.close()
    print(f"{spec['id']}: {len(store)} total steps")

print(store)

## Optional: push to the Hugging Face Hub

Set `REPO_ID` (e.g. `your-org/my-mouse-dataset`) to upload.

In [ ]:
REPO_ID = "mouse-core-test"  # e.g. "your-org/my-mouse-dataset"

if REPO_ID:
    url = push_stores_to_hub([store], repo_id=REPO_ID, split="train", private=False)
    print(f"Pushed to {url}")
else:
    print("Set REPO_ID to upload this dataset to the Hugging Face Hub.")